# SONA — VoiceBank-DEMAND benchmark (run-all)

Classical baselines + supervised learned baselines (MaskGRU / MaskTCN / Attention) + RL frozen
vs **RL online (SQUIM reward)** on the VoiceBank-DEMAND test set (its noises + speakers are unseen
in training → a built-in distribution-shift test).

### Before you Run All (one-time, ~2 min)
1. **Add Data** (right panel): attach the **`sona-repo`** dataset (your uploaded `sona.zip`) and a
   **VoiceBank-DEMAND** dataset that includes the *train* folders
   (`clean_trainset_wav`, `noisy_trainset_wav`, `clean_testset_wav`, `noisy_testset_wav`).
2. **Settings**: Accelerator = **GPU T4**, Internet = **On** (needed for `pip install pesq` + SQUIM download).
3. **Run All**. Paths are auto-detected; tune the knobs in the parameters cell if you want.


In [ ]:
# 1) Deps. torch/torchaudio are preinstalled on Kaggle (SQUIM lives in torchaudio).
%pip install -q pesq pystoi ptflops
import torch, torchaudio
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2) Auto-detect the repo and the VoiceBank-DEMAND folder among the attached datasets (any depth).
import glob, os

def find_repo():
    hits = glob.glob('/kaggle/input/**/scripts/kaggle_run.py', recursive=True)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None

def find_vbd():
    hits = glob.glob('/kaggle/input/**/noisy_trainset_wav', recursive=True)
    return os.path.dirname(hits[0]) if hits else None

REPO_DIR = find_repo()
VBD_ROOT = find_vbd()
OUT_DIR  = '/kaggle/working/results'

# If auto-detect fails, hard-code from the paths you see in the right-hand panel:
# REPO_DIR = '/kaggle/input/sona-repo'
# VBD_ROOT = '/kaggle/input/voicebank-demand-16k'

assert REPO_DIR, 'sona-repo not found — attach it (must contain scripts/kaggle_run.py)'
assert VBD_ROOT, 'VoiceBank-DEMAND not found — attach a dataset with noisy_trainset_wav/'
print('REPO_DIR =', REPO_DIR)
print('VBD_ROOT =', VBD_ROOT)

In [ ]:
# 3) Run parameters. Defaults give a real first result in ~1-1.5 h on a T4.
#    Epoch calibration on real VBD shows the small models plateau by ~epoch 25-30, so 40 is a safe cap.
LIMIT_TRAIN = 2000      # raise to e.g. 8000 or None for the full 11,572 train set (slower, stronger)
LIMIT_TEST  = 300       # set to 0 to use the whole 824-clip test set (needed for a paper number)
EPOCHS      = 40        # supervised epochs (per learned baseline)
RL_EPOCHS   = 40
SUP_MODELS  = 'mask_gru,mask_tcn,attention'  # supervised learned baselines; trim to e.g. 'mask_gru,attention' to save time
ABLATE      = False     # True -> also eval scalar/lowrank/band_bias/film online adapters (slower)
print(dict(LIMIT_TRAIN=LIMIT_TRAIN, LIMIT_TEST=LIMIT_TEST, EPOCHS=EPOCHS, RL_EPOCHS=RL_EPOCHS, SUP_MODELS=SUP_MODELS, ABLATE=ABLATE))

In [ ]:
# 4) Run the full benchmark (classical + supervised baselines + RL frozen vs RL online/SQUIM).
import subprocess, sys
cmd = [sys.executable, f'{REPO_DIR}/scripts/kaggle_run.py',
       '--repo', REPO_DIR, '--vbd-root', VBD_ROOT, '--out', OUT_DIR,
       '--limit-train', str(LIMIT_TRAIN), '--epochs', str(EPOCHS), '--rl-epochs', str(RL_EPOCHS),
       '--sup-models', SUP_MODELS]
if LIMIT_TEST:
    cmd += ['--limit-test', str(LIMIT_TEST)]
if ABLATE:
    cmd += ['--ablate-adapters']
print(' '.join(cmd), '\n')
# Stream output live.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'kaggle_run.py exited {proc.returncode}'

In [ ]:
# 5) Show the saved results table.
import pandas as pd
from pathlib import Path
print(Path(OUT_DIR, 'summary.md').read_text())
pd.read_csv(f'{OUT_DIR}/summary.csv')